# Inspect the Saved FAISS Vector Database

Use this notebook to verify how many documents are stored in the existing FAISS index and to preview their metadata and content snippets.

In [14]:
from __future__ import annotations

import random
from collections import Counter
from pathlib import Path
from typing import List

import torch
from langchain.embeddings import SentenceTransformerEmbeddings
from langchain.vectorstores import FAISS

In [ ]:
INDEX_DIR = Path("faiss_index_backup/backup_713450")
INDEX_NAME = "index"
EMBEDDING_MODEL = "all-MiniLM-L6-v2"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
ALLOW_DANGEROUS_DESERIALIZATION = True
SAMPLE_SIZE = 5
USE_RANDOM_SAMPLE = True
QUERY = ""
TOP_K = 3

In [16]:
if not INDEX_DIR.exists():
    raise FileNotFoundError(f"Vector index directory '{INDEX_DIR}' was not found.")

embeddings = SentenceTransformerEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={"device": DEVICE},
)

vector_db = FAISS.load_local(
    INDEX_DIR,
    embeddings,
    index_name=INDEX_NAME,
    allow_dangerous_deserialization=ALLOW_DANGEROUS_DESERIALIZATION,
)

vector_count = vector_db.index.ntotal
print(f"Loaded vector database from {INDEX_DIR} with {vector_count} vectors")

Loaded vector database from faiss_index with 136350 vectors


In [17]:
docstore_dict = vector_db.docstore._dict
doc_ids: List[str] = list(docstore_dict.keys())
documents = [docstore_dict[doc_id] for doc_id in doc_ids]

doi_counter = Counter(doc.metadata.get("doi", "Unknown DOI") for doc in documents)
source_counter = Counter(doc.metadata.get("source", "unknown") for doc in documents)

print(f"Stored documents: {len(documents)}")
print(f"Unique DOIs: {len(doi_counter)}")
print(f"Sources: {source_counter}")

Stored documents: 136350
Unique DOIs: 4106
Sources: Counter({'paragraph': 132120, 'abstract': 4230})


In [18]:
if not documents:
    print("No documents available in the vector store.")
else:
    sample_count = min(SAMPLE_SIZE, len(documents))
    sample_docs = random.sample(documents, sample_count) if USE_RANDOM_SAMPLE else documents[:sample_count]
    
    for idx, doc in enumerate(sample_docs, start=1):
        print(f"\n--- Sample Document {idx} ---")
        print(f"DOI: {doc.metadata.get('doi', 'Unknown DOI')}")
        print(f"Source: {doc.metadata.get('source', 'unknown')}")
        preview = doc.page_content.replace("\n", " ").strip()
        snippet = preview[:400]
        if len(preview) > 400:
            snippet += "..."
        print(f"Content preview: {snippet}")


--- Sample Document 1 ---
DOI: 10.1016/j.micromeso.2007.05.066
Source: paragraph
Content preview: On the basis of the results depicted in Fig. 4b, it appears that adsorption of oxygen in zeolite at ambient temperature does not enable NO2 to be formed. Contrarily special active sites in the zeolite are desirable for the formation of NO2. These sites were deactivated during the TPSR process and some of them cannot be regenerated. Two possible reactions may give rise to NO2 in the TPSR process of...

--- Sample Document 2 ---
DOI: 10.1038/s41467-018-04748-x
Source: paragraph
Content preview: Palladium supported on zeolite is a highly active material for complete methane oxidation2,15,20,21. Zeolites offer several advantages compared to metal oxide supports, the most fascinating one being the possibility to constrain small metal particles to protect them from sintering22,23, which is otherwise achieved upon laborious chemical treatment1,10,11. This opportunity has not been exploited so...

In [19]:
if QUERY:
    results = vector_db.similarity_search(QUERY, k=TOP_K)
    if not results:
        print("No documents retrieved for the supplied query.")
    else:
        print(f"\nTop {len(results)} results for query: {QUERY!r}")
        for rank, doc in enumerate(results, start=1):
            print(f"\nResult {rank}")
            print(f"DOI: {doc.metadata.get('doi', 'Unknown DOI')}")
            print(f"Source: {doc.metadata.get('source', 'unknown')}")
            preview = doc.page_content.replace("\n", " ").strip()
            snippet = preview[:400]
            if len(preview) > 400:
                snippet += "..."
            print(f"Content preview: {snippet}")